# EDA on Online Retail Sales

**Oasis Infobyte — Data Analytics Level 1, Task 1**

This notebook explores the Online Retail transaction dataset through data-quality checks, cleaning, descriptive statistics, time trends, market analysis, product analysis, correlation analysis and business recommendations.

> **Data note:** Age and gender are not present in this dataset, so they are not analysed. Country is used as the available market dimension.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
BASE = Path.cwd()
candidates = list((BASE/'data').rglob('*.csv')) + list((BASE/'data').rglob('*.xlsx'))
if not candidates:
    raise FileNotFoundError('No CSV/XLSX dataset found under the project data folder.')
DATA = candidates[0]
OUT = BASE/'outputs'
OUT.mkdir(exist_ok=True)
df_raw = pd.read_excel(DATA) if DATA.suffix.lower()=='.xlsx' else pd.read_csv(DATA, encoding='latin1')
print('Dataset:', DATA)
print('Shape:', df_raw.shape)
display(df_raw.head())

## 1. Initial inspection

In [ ]:
display(df_raw.dtypes.rename('dtype').to_frame())
display(df_raw.isna().sum().sort_values(ascending=False).rename('missing_values').to_frame())
print('Duplicate rows:', int(df_raw.duplicated().sum()))
display(df_raw.describe(include='all').T)

## 2. Data cleaning

Exact duplicates are removed. Dates are converted to datetime, text fields are trimmed, cancelled invoices and non-positive sales are excluded, and Revenue is calculated as Quantity × UnitPrice.

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.strip()
for col in ['Description','Country','StockCode','InvoiceNo']:
    if col in df.columns: df[col] = df[col].astype('string').str.strip()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
before_rows = len(df)
before_duplicates = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
df['is_cancelled'] = df['InvoiceNo'].astype('string').str.upper().str.startswith('C', na=False)
df['Revenue'] = df['Quantity'] * df['UnitPrice']
sales_df = df[(~df['is_cancelled']) & (df['Quantity'] > 0) & (df['UnitPrice'] > 0) & df['InvoiceDate'].notna()].copy()
print(f'Rows before cleaning: {before_rows:,}')
print(f'Duplicate rows removed: {before_duplicates:,}')
print(f'Rows after cleaning: {len(sales_df):,}')
sales_df.to_csv(OUT/'cleaned_sales.csv', index=False)

## 3. Descriptive statistics

In [ ]:
display(sales_df[['Quantity','UnitPrice','Revenue']].describe().T)
print('Unique invoices:', sales_df['InvoiceNo'].nunique())
print('Unique products:', sales_df['Description'].nunique())
print('Unique countries:', sales_df['Country'].nunique())
print('Total revenue: £{:,.2f}'.format(sales_df['Revenue'].sum()))

## 4. Monthly and quarterly sales trends

In [ ]:
sales_df['Month'] = sales_df['InvoiceDate'].dt.to_period('M').astype(str)
sales_df['Quarter'] = sales_df['InvoiceDate'].dt.to_period('Q').astype(str)
monthly = sales_df.groupby('Month')['Revenue'].sum()
quarterly = sales_df.groupby('Quarter')['Revenue'].sum()
fig, ax = plt.subplots(figsize=(12,5)); monthly.plot(marker='o', ax=ax); ax.set_title('Monthly Revenue Trend'); ax.set_xlabel('Month'); ax.set_ylabel('Revenue (£)'); plt.xticks(rotation=45); plt.tight_layout(); plt.savefig(OUT/'monthly_revenue_trend.png', dpi=160); plt.show()
fig, ax = plt.subplots(figsize=(11,5)); quarterly.plot(marker='o', ax=ax); ax.set_title('Quarterly Revenue Trend'); ax.set_xlabel('Quarter'); ax.set_ylabel('Revenue (£)'); plt.tight_layout(); plt.savefig(OUT/'quarterly_revenue_trend.png', dpi=160); plt.show()
print('Peak month:', monthly.idxmax(), '— £{:,.2f}'.format(monthly.max()))
print('Peak quarter:', quarterly.idxmax(), '— £{:,.2f}'.format(quarterly.max()))

### Key visualisations

![Monthly Revenue Trend](outputs/monthly_revenue_trend.png)

![Quarterly Revenue Trend](outputs/quarterly_revenue_trend.png)

## 5. Market analysis

In [ ]:
country_revenue = sales_df.groupby('Country')['Revenue'].sum().sort_values(ascending=False)
country_orders = sales_df.groupby('Country')['InvoiceNo'].nunique().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10,6)); country_revenue.head(10).sort_values().plot(kind='barh', ax=ax); ax.set_title('Top 10 Countries by Revenue'); ax.set_xlabel('Revenue (£)'); plt.tight_layout(); plt.savefig(OUT/'top_10_countries_by_revenue.png', dpi=160); plt.show()
fig, ax = plt.subplots(figsize=(10,6)); country_orders.head(10).sort_values().plot(kind='barh', ax=ax); ax.set_title('Top 10 Countries by Number of Orders'); ax.set_xlabel('Unique invoices'); plt.tight_layout(); plt.savefig(OUT/'top_10_countries_by_orders.png', dpi=160); plt.show()

### Country analysis visualisations

![Top 10 Countries by Revenue](outputs/top_10_countries_by_revenue.png)

![Top 10 Countries by Orders](outputs/top_10_countries_by_orders.png)

## 6. Product analysis

In [ ]:
top_products = sales_df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
display(top_products.to_frame('units_sold'))
fig, ax = plt.subplots(figsize=(10,6)); top_products.sort_values().plot(kind='barh', ax=ax); ax.set_title('Top 10 Products by Units Sold'); ax.set_xlabel('Units sold'); plt.tight_layout(); plt.savefig(OUT/'top_10_products_by_units.png', dpi=160); plt.show()

### Top 10 products by units sold

![Top 10 Products by Units Sold](outputs/top_10_products_by_units.png)

## 7. Correlation heatmap

In [ ]:
corr = sales_df[['Quantity','UnitPrice','Revenue']].corr()
plt.figure(figsize=(8,6)); sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm'); plt.title('Correlation Matrix'); plt.tight_layout(); plt.savefig(OUT/'correlation_heatmap.png', dpi=160); plt.show()

### Correlation matrix

![Correlation Heatmap](outputs/correlation_heatmap.png)

## 8. Insights and recommendations

The analysis highlights when revenue peaks, which markets contribute most revenue, which products have the highest unit demand, and how the numeric sales variables relate.

**Recommendations:** plan inventory and promotions around peak periods; protect availability of high-volume products and test bundles; focus retention and localisation efforts on major markets while selectively developing smaller markets; and improve CustomerID completeness for stronger customer-level retention and lifetime-value analysis.

In [ ]:
total_revenue = sales_df['Revenue'].sum()
top10_share = country_revenue.head(10).sum() / total_revenue
print(f'Top-10-country revenue share: {top10_share:.1%}')
if 'CustomerID' in sales_df.columns:
    customer_summary = sales_df.dropna(subset=['CustomerID']).groupby('CustomerID').agg(Orders=('InvoiceNo','nunique'), Revenue=('Revenue','sum'), Units=('Quantity','sum')).sort_values('Revenue', ascending=False)
    display(customer_summary.head(10))

## 9. Conclusion

After cleaning, the dataset provides a reliable basis for transaction-level retail analysis. The strongest decision areas are demand timing, product availability, market concentration and improving customer-level data completeness.